# Notebook 14 — Trajectory Grading

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/evals/14_trajectory_grading.ipynb)

This notebook moves from outcome scoring to trace inspection. We will grade whole trajectories step by step, detect loops and stalls, and evaluate whether an agent terminated for the right reason.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **What you will build**
- Understand **Why trajectory grading matters**
- Understand **Step 1 — Shared helpers**
- Understand **Step 2 — Define the trace schema**
- Understand **Step 3 — Inspect label balance**


## What you will build

- a compact benchmark of labeled agent traces
- step-level audit functions for usefulness, argument completeness, and progress
- loop and stall detectors
- a termination-quality scorer
- structured audit artifacts for later regression tests

## Why trajectory grading matters

Two trajectories can end with the same final answer while behaving very differently. One may verify facts and stop cleanly. Another may repeat tools, skip critical checks, or declare success too early.

Trajectory grading makes those differences measurable.

In [ ]:
# --- Google Colab Setup ---
!pip install -q "transformers>=4.51.0" accelerate bitsandbytes torch

import csv
import json
import math
import random
import statistics
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True, top_k=20):
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=top_k,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated_ids = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## Step 1 — Shared helpers

The full notebook uses plain Python data structures so every score is transparent and editable.

In [ ]:
from collections import Counter, defaultdict
from pathlib import Path
import csv
import json
import random
import statistics

random.seed(14)

ARTIFACT_DIR = Path("artifacts") / "notebook_14_trajectory_grading"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def clip(text, width=78):
    text = str(text)
    return text if len(text) <= width else text[: width - 3] + "..."


def normalize_text(text):
    cleaned = "".join(char.lower() if char.isalnum() else " " for char in str(text))
    return " ".join(cleaned.split())


def print_rows(rows, columns=None, limit=None):
    rows = list(rows)
    if limit is not None:
        rows = rows[:limit]
    if not rows:
        print("(no rows)")
        return
    if columns is None:
        columns = list(rows[0].keys())
    widths = {}
    for column in columns:
        widths[column] = len(str(column))
        for row in rows:
            widths[column] = max(widths[column], len(str(row.get(column, ""))))
    header = " | ".join(str(column).ljust(widths[column]) for column in columns)
    divider = "-+-".join("-" * widths[column] for column in columns)
    print(header)
    print(divider)
    for row in rows:
        print(" | ".join(str(row.get(column, "")).ljust(widths[column]) for column in columns))


print("Artifact directory:", ARTIFACT_DIR.resolve())

## Step 2 — Define the trace schema

Each trace contains:

- a task description
- required tools and required facts
- gold issue labels
- a list of steps with explicit kind and new_facts fields

That gives us enough structure to build graders without hidden heuristics.

In [ ]:
TRACE_BENCHMARK = [
    {
        "trace_id": "trace_refund_clean",
        "task": "Resolve a duplicate charge after verifying the order.",
        "required_tools": ["lookup_order", "issue_refund"],
        "required_facts": ["order_verified", "refund_issued"],
        "labels": {"loop": False, "stall": False, "bad_termination": False},
        "steps": [
            {"kind": "plan", "text": "Verify the order, then refund the duplicate charge.", "new_facts": []},
            {"kind": "tool", "tool": "lookup_order", "args": {"order_id": "O-9001"}, "new_facts": ["order_verified"]},
            {"kind": "tool", "tool": "issue_refund", "args": {"order_id": "O-9001", "amount": 44.0}, "new_facts": ["refund_issued"]},
            {"kind": "final", "status": "completed", "text": "Refund completed and customer notified.", "new_facts": []},
        ],
    },
    {
        "trace_id": "trace_reset_premature_finish",
        "task": "Investigate a password reset outage and open the right ticket.",
        "required_tools": ["search_docs", "create_ticket"],
        "required_facts": ["doc_reviewed", "ticket_created"],
        "labels": {"loop": False, "stall": False, "bad_termination": True},
        "steps": [
            {"kind": "plan", "text": "Check whether the reset outage is already documented.", "new_facts": []},
            {"kind": "tool", "tool": "search_docs", "args": {"query": "password reset 404"}, "new_facts": ["doc_reviewed"]},
            {"kind": "final", "status": "completed", "text": "This looks like a known issue, so I am done.", "new_facts": []},
        ],
    },
    {
        "trace_id": "trace_outage_loop",
        "task": "Open a network outage ticket for Building 7.",
        "required_tools": ["search_docs", "create_ticket"],
        "required_facts": ["doc_reviewed", "ticket_created"],
        "labels": {"loop": True, "stall": False, "bad_termination": False},
        "steps": [
            {"kind": "plan", "text": "Search for context before filing the incident.", "new_facts": []},
            {"kind": "tool", "tool": "search_docs", "args": {"query": "building 7 outage"}, "new_facts": []},
            {"kind": "tool", "tool": "search_docs", "args": {"query": "building 7 outage"}, "new_facts": []},
            {"kind": "tool", "tool": "search_docs", "args": {"query": "building 7 outage"}, "new_facts": ["doc_reviewed"]},
            {"kind": "tool", "tool": "create_ticket", "args": {"queue": "network_ops", "priority": "critical"}, "new_facts": ["ticket_created"]},
            {"kind": "final", "status": "completed", "text": "Ticket created after repeated searching.", "new_facts": []},
        ],
    },
    {
        "trace_id": "trace_meter_stall",
        "task": "Book a safety visit for a sparking meter.",
        "required_tools": ["create_ticket", "schedule_visit"],
        "required_facts": ["ticket_created", "visit_booked"],
        "labels": {"loop": False, "stall": True, "bad_termination": False},
        "steps": [
            {"kind": "plan", "text": "I should probably do something about this meter.", "new_facts": []},
            {"kind": "plan", "text": "Still thinking about the safest next step.", "new_facts": []},
            {"kind": "tool", "tool": "create_ticket", "args": {"queue": "field_safety", "priority": "urgent"}, "new_facts": ["ticket_created"]},
            {"kind": "plan", "text": "Maybe I should ask again before scheduling.", "new_facts": []},
            {"kind": "plan", "text": "No new information yet.", "new_facts": []},
            {"kind": "final", "status": "blocked", "text": "Ticket exists but no visit was scheduled.", "new_facts": []},
        ],
    },
    {
        "trace_id": "trace_privacy_clean",
        "task": "Handle a privacy export request through the correct team.",
        "required_tools": ["search_docs", "escalate_case"],
        "required_facts": ["doc_reviewed", "case_escalated"],
        "labels": {"loop": False, "stall": False, "bad_termination": False},
        "steps": [
            {"kind": "plan", "text": "Check the privacy workflow and escalate with the required reason.", "new_facts": []},
            {"kind": "tool", "tool": "search_docs", "args": {"query": "data export privacy"}, "new_facts": ["doc_reviewed"]},
            {"kind": "tool", "tool": "escalate_case", "args": {"team": "privacy_ops", "reason": "data_export"}, "new_facts": ["case_escalated"]},
            {"kind": "final", "status": "completed", "text": "Privacy case opened and acknowledgment sent.", "new_facts": []},
        ],
    },
    {
        "trace_id": "trace_billing_duplicate_lookup",
        "task": "Refund a canceled add-on after order verification.",
        "required_tools": ["lookup_order", "issue_refund"],
        "required_facts": ["order_verified", "refund_issued"],
        "labels": {"loop": True, "stall": False, "bad_termination": False},
        "steps": [
            {"kind": "plan", "text": "Verify the order before refunding.", "new_facts": []},
            {"kind": "tool", "tool": "lookup_order", "args": {"order_id": "O-9014"}, "new_facts": ["order_verified"]},
            {"kind": "tool", "tool": "lookup_order", "args": {"order_id": "O-9014"}, "new_facts": []},
            {"kind": "tool", "tool": "issue_refund", "args": {"order_id": "O-9014", "amount": 19.0}, "new_facts": ["refund_issued"]},
            {"kind": "final", "status": "completed", "text": "Refund completed after duplicate lookup.", "new_facts": []},
        ],
    },
    {
        "trace_id": "trace_shipping_false_complete",
        "task": "Send a delayed-shipment update after verifying the order.",
        "required_tools": ["lookup_order", "send_message"],
        "required_facts": ["order_verified", "message_sent"],
        "labels": {"loop": False, "stall": False, "bad_termination": True},
        "steps": [
            {"kind": "plan", "text": "Check the order and then reply.", "new_facts": []},
            {"kind": "tool", "tool": "lookup_order", "args": {"order_id": ""}, "new_facts": []},
            {"kind": "final", "status": "completed", "text": "The delay has been explained to the customer.", "new_facts": []},
        ],
    },
    {
        "trace_id": "trace_field_never_finishes",
        "task": "Schedule maintenance for asset A-12 and confirm the slot.",
        "required_tools": ["schedule_visit", "send_message"],
        "required_facts": ["visit_booked", "message_sent"],
        "labels": {"loop": True, "stall": True, "bad_termination": True},
        "steps": [
            {"kind": "plan", "text": "Need to book the afternoon slot.", "new_facts": []},
            {"kind": "tool", "tool": "schedule_visit", "args": {"asset_id": "A-12", "slot": "2026-04-10 09:00"}, "new_facts": []},
            {"kind": "tool", "tool": "schedule_visit", "args": {"asset_id": "A-12", "slot": "2026-04-10 09:00"}, "new_facts": []},
            {"kind": "tool", "tool": "schedule_visit", "args": {"asset_id": "A-12", "slot": "2026-04-10 09:00"}, "new_facts": []},
            {"kind": "plan", "text": "Still waiting for a better slot.", "new_facts": []},
        ],
    },
]

print_rows(
    [
        {
            "trace_id": trace["trace_id"],
            "task": clip(trace["task"], 55),
            "steps": len(trace["steps"]),
            "labels": trace["labels"],
        }
        for trace in TRACE_BENCHMARK
    ],
    columns=["trace_id", "task", "steps", "labels"],
)

## Step 3 — Inspect label balance

Issue detection works best when you know how many positive cases exist for each label.

In [ ]:
label_counts = Counter()
for trace in TRACE_BENCHMARK:
    for label, value in trace["labels"].items():
        if value:
            label_counts[label] += 1

print("Positive label counts:", dict(label_counts))
print("Total traces:", len(TRACE_BENCHMARK))

## Step 4 — Tool contracts for step grading

Argument completeness is one of the cheapest and most reliable trajectory signals. We define the required fields once and reuse them in every audit.

In [ ]:
TOOL_REQUIREMENTS = {
    "search_docs": ("query",),
    "lookup_order": ("order_id",),
    "issue_refund": ("order_id", "amount"),
    "create_ticket": ("queue", "priority"),
    "schedule_visit": ("asset_id", "slot"),
    "escalate_case": ("team", "reason"),
    "send_message": ("template",),
}

print_rows(
    [{"tool": tool, "required_args": args} for tool, args in TOOL_REQUIREMENTS.items()],
    columns=["tool", "required_args"],
)

## Step 5 — Build a step-level audit function

The auditor keeps track of:

- which required tools have been seen
- which facts have been unlocked
- whether the current step adds progress
- whether the step repeats an earlier action without new evidence

In [ ]:
def step_signature(step):
    if step["kind"] != "tool":
        return (step["kind"], normalize_text(step.get("text", "")))
    args = tuple(sorted((key, str(value)) for key, value in step.get("args", {}).items()))
    return (step["tool"], args)


def audit_trace(trace):
    seen_tools = []
    seen_facts = set()
    audits = []

    for index, step in enumerate(trace["steps"], start=1):
        remaining_tools = [tool for tool in trace["required_tools"] if tool not in seen_tools]
        new_facts = [fact for fact in step.get("new_facts", []) if fact not in seen_facts]
        progress = 1 if new_facts else 0
        args_complete = 1.0
        expected_tool = 0
        repeated = 0

        if step["kind"] == "tool":
            required_args = TOOL_REQUIREMENTS.get(step["tool"], ())
            if required_args:
                present = sum(1 for arg in required_args if step.get("args", {}).get(arg) not in (None, ""))
                args_complete = present / len(required_args)
            expected_tool = 1 if step["tool"] in remaining_tools else 0
            repeated = int(
                any(
                    earlier["kind"] == "tool" and step_signature(earlier) == step_signature(step)
                    for earlier in trace["steps"][: index - 1]
                )
            )
            seen_tools.append(step["tool"])
        elif step["kind"] == "plan":
            plan_text = normalize_text(step.get("text", ""))
            expected_tool = int(any(normalize_text(tool.split("_")[0]) in plan_text for tool in remaining_tools))

        seen_facts.update(new_facts)
        usefulness = round(0.45 * progress + 0.35 * args_complete + 0.20 * expected_tool, 3)

        audits.append(
            {
                "trace_id": trace["trace_id"],
                "step_index": index,
                "kind": step["kind"],
                "tool": step.get("tool", ""),
                "status": step.get("status", ""),
                "progress": progress,
                "args_complete": round(args_complete, 3),
                "expected_tool": expected_tool,
                "repeated": repeated,
                "new_facts": len(new_facts),
                "usefulness": usefulness,
            }
        )

    return audits

## Step 6 — Audit one trace by hand

A quick manual inspection helps confirm that the audit logic matches our intuition before we summarize the whole benchmark.

In [ ]:
demo_trace = next(trace for trace in TRACE_BENCHMARK if trace["trace_id"] == "trace_outage_loop")
demo_audit = audit_trace(demo_trace)
print_rows(
    demo_audit,
    columns=["step_index", "kind", "tool", "progress", "args_complete", "expected_tool", "repeated", "usefulness"],
)

## Step 7 — Detect loops and stalls

We will implement two simple detectors:

- **loop**: repeated tool signatures with no new facts
- **stall**: a long streak of non-final steps with no progress

In [ ]:
def detect_loop(audits):
    repeated_no_progress = [
        row for row in audits
        if row["kind"] == "tool" and row["repeated"] == 1 and row["progress"] == 0
    ]
    return len(repeated_no_progress) >= 1


def detect_stall(audits):
    longest_streak = 0
    current = 0
    for row in audits:
        if row["kind"] != "final" and row["progress"] == 0:
            current += 1
            longest_streak = max(longest_streak, current)
        else:
            current = 0
    return longest_streak >= 3, longest_streak


loop_demo = detect_loop(demo_audit)
stall_demo, streak_demo = detect_stall(demo_audit)
print("Loop detected:", loop_demo)
print("Stall detected:", stall_demo, "with streak", streak_demo)

## Step 8 — Grade termination quality

Termination is good when the final step matches the state of the work. Declaring completion too early or never emitting a final status should be penalized.

In [ ]:
def grade_termination(trace, audits):
    final_steps = [step for step in trace["steps"] if step["kind"] == "final"]
    seen_tools_before_final = set()
    seen_facts_before_final = set()
    for step in trace["steps"]:
        if step["kind"] == "final":
            break
        if step["kind"] == "tool":
            seen_tools_before_final.add(step["tool"])
        seen_facts_before_final.update(step.get("new_facts", []))

    missing_tools = [tool for tool in trace["required_tools"] if tool not in seen_tools_before_final]
    missing_facts = [fact for fact in trace["required_facts"] if fact not in seen_facts_before_final]

    if not final_steps:
        return {
            "termination_score": 0.0,
            "termination_label": "missing_final",
            "missing_tools": missing_tools,
            "missing_facts": missing_facts,
        }

    final_step = final_steps[-1]
    if final_step["status"] == "completed" and (missing_tools or missing_facts):
        return {
            "termination_score": 0.0,
            "termination_label": "premature_complete",
            "missing_tools": missing_tools,
            "missing_facts": missing_facts,
        }
    if final_step["status"] == "blocked" and missing_tools:
        return {
            "termination_score": 0.7,
            "termination_label": "reasonable_block",
            "missing_tools": missing_tools,
            "missing_facts": missing_facts,
        }
    return {
        "termination_score": 1.0,
        "termination_label": "clean_finish",
        "missing_tools": missing_tools,
        "missing_facts": missing_facts,
    }

## Step 9 — Grade all trajectories

The full grader combines step usefulness, coverage of required tools and facts, loop and stall penalties, and termination quality.

In [ ]:
trace_summaries = []
all_step_audits = []

for trace in TRACE_BENCHMARK:
    audits = audit_trace(trace)
    all_step_audits.extend(audits)
    loop_flag = detect_loop(audits)
    stall_flag, longest_streak = detect_stall(audits)
    termination = grade_termination(trace, audits)

    seen_tools = {step.get("tool") for step in trace["steps"] if step["kind"] == "tool"}
    seen_facts = set()
    for step in trace["steps"]:
        seen_facts.update(step.get("new_facts", []))

    tool_coverage = len(set(trace["required_tools"]) & seen_tools) / len(trace["required_tools"])
    fact_coverage = len(set(trace["required_facts"]) & seen_facts) / len(trace["required_facts"])
    mean_usefulness = statistics.fmean(row["usefulness"] for row in audits)
    grade = round(
        0.30 * mean_usefulness
        + 0.25 * tool_coverage
        + 0.20 * fact_coverage
        + 0.25 * termination["termination_score"]
        - (0.10 if loop_flag else 0.0)
        - (0.10 if stall_flag else 0.0),
        3,
    )

    trace_summaries.append(
        {
            "trace_id": trace["trace_id"],
            "steps": len(trace["steps"]),
            "mean_usefulness": round(mean_usefulness, 3),
            "tool_coverage": round(tool_coverage, 3),
            "fact_coverage": round(fact_coverage, 3),
            "loop_pred": int(loop_flag),
            "stall_pred": int(stall_flag),
            "termination_label": termination["termination_label"],
            "termination_score": termination["termination_score"],
            "longest_no_progress_streak": longest_streak,
            "trajectory_grade": grade,
        }
    )

print_rows(
    trace_summaries,
    columns=[
        "trace_id",
        "steps",
        "mean_usefulness",
        "tool_coverage",
        "fact_coverage",
        "loop_pred",
        "stall_pred",
        "termination_label",
        "trajectory_grade",
    ],
)

## Step 10 — Evaluate detector quality against labels

Because the benchmark includes gold labels, we can score the detectors directly instead of trusting a few anecdotal examples.

In [ ]:
def detector_metrics(label_name, prediction_key):
    gold = []
    pred = []
    for trace, summary in zip(TRACE_BENCHMARK, trace_summaries):
        gold.append(int(trace["labels"][label_name]))
        pred.append(int(summary[prediction_key]))

    tp = sum(1 for g, p in zip(gold, pred) if g == 1 and p == 1)
    fp = sum(1 for g, p in zip(gold, pred) if g == 0 and p == 1)
    fn = sum(1 for g, p in zip(gold, pred) if g == 1 and p == 0)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "label": label_name,
        "precision": round(precision, 3),
        "recall": round(recall, 3),
        "f1": round(f1, 3),
        "tp": tp,
        "fp": fp,
        "fn": fn,
    }


detector_report = [
    detector_metrics("loop", "loop_pred"),
    detector_metrics("stall", "stall_pred"),
]

termination_rows = []
for trace, summary in zip(TRACE_BENCHMARK, trace_summaries):
    predicted_bad_termination = int(summary["termination_label"] in {"premature_complete", "missing_final"})
    gold_bad_termination = int(trace["labels"]["bad_termination"])
    termination_rows.append({"gold": gold_bad_termination, "pred": predicted_bad_termination})

tp = sum(1 for row in termination_rows if row["gold"] == 1 and row["pred"] == 1)
fp = sum(1 for row in termination_rows if row["gold"] == 0 and row["pred"] == 1)
fn = sum(1 for row in termination_rows if row["gold"] == 1 and row["pred"] == 0)
precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn) if tp + fn else 0.0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
detector_report.append(
    {
        "label": "bad_termination",
        "precision": round(precision, 3),
        "recall": round(recall, 3),
        "f1": round(f1, 3),
        "tp": tp,
        "fp": fp,
        "fn": fn,
    }
)

print_rows(detector_report, columns=["label", "precision", "recall", "f1", "tp", "fp", "fn"])

## Step 11 — Bucket the audit failures

A useful trace notebook ends with actionable categories, not only scalar scores.

In [ ]:
failure_buckets = Counter()
for trace, summary in zip(TRACE_BENCHMARK, trace_summaries):
    if summary["loop_pred"]:
        failure_buckets["loop"] += 1
    if summary["stall_pred"]:
        failure_buckets["stall"] += 1
    if summary["termination_label"] == "premature_complete":
        failure_buckets["premature_complete"] += 1
    if summary["termination_label"] == "missing_final":
        failure_buckets["missing_final"] += 1
    if summary["tool_coverage"] < 1.0:
        failure_buckets["missing_required_tool"] += 1
    if summary["fact_coverage"] < 1.0:
        failure_buckets["missing_required_fact"] += 1

bucket_rows = [{"bucket": bucket, "count": count} for bucket, count in sorted(failure_buckets.items())]
print_rows(bucket_rows, columns=["bucket", "count"])

## Step 12 — Review the lowest-graded traces

When an eval fails, the first operator question is usually: which traces deserve inspection first?

In [ ]:
lowest_grade_rows = sorted(trace_summaries, key=lambda row: row["trajectory_grade"])[:4]
print_rows(
    lowest_grade_rows,
    columns=[
        "trace_id",
        "trajectory_grade",
        "tool_coverage",
        "fact_coverage",
        "termination_label",
        "loop_pred",
        "stall_pred",
    ],
)

## Step 13 — Save structured audit artifacts

We will write both trajectory-level and step-level outputs so the notebook can feed later regression and reporting workflows.

In [ ]:
def write_csv(path, rows):
    rows = list(rows)
    if not rows:
        return
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


(ARTIFACT_DIR / "trace_summaries.json").write_text(
    json.dumps(trace_summaries, indent=2),
    encoding="utf-8",
)
(ARTIFACT_DIR / "detector_report.json").write_text(
    json.dumps(detector_report, indent=2),
    encoding="utf-8",
)
write_csv(ARTIFACT_DIR / "trace_summaries.csv", trace_summaries)
write_csv(ARTIFACT_DIR / "step_audits.csv", all_step_audits)

print("Saved artifacts:")
for path in sorted(ARTIFACT_DIR.iterdir()):
    print("-", path.name)

## Step 14 — Create an operator-facing summary

This final report translates audit measurements into debugging priorities.

In [ ]:
report = {
    "average_grade": round(statistics.fmean(row["trajectory_grade"] for row in trace_summaries), 3),
    "worst_trace": min(trace_summaries, key=lambda row: row["trajectory_grade"])["trace_id"],
    "best_trace": max(trace_summaries, key=lambda row: row["trajectory_grade"])["trace_id"],
    "detector_report": detector_report,
    "top_failure_buckets": sorted(bucket_rows, key=lambda row: row["count"], reverse=True)[:3],
    "recommendations": [
        "Penalize repeated zero-progress tool calls directly in the agent reward or rubric.",
        "Require an explicit final status so missing termination becomes machine-detectable.",
        "Track fact coverage separately from tool coverage to catch premature completion.",
    ],
}

report_path = ARTIFACT_DIR / "operator_report.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))

## Recap

You now have a first-principles trajectory grader that can:

- audit each step in a trace
- detect loops and stalls
- judge termination quality
- save structured artifacts for later analysis

This is the bridge between simple tool-use evals and full multi-agent workflow evaluation.

## 🏋️ Exercises

Put your understanding to the test:

**Exercise 1 — Explore:** design an eval dataset for a new use case. Document what changes and why.

**Exercise 2 — Build:** implement a custom metric and compare it to the one in this notebook. Compare your implementation to the one in this notebook.

**Exercise 3 — Extend:** run the evaluation on a different model and analyze the differences.

## 📚 References & Further Reading

- [Hugging Face Documentation](https://huggingface.co/docs)
- [LLM Course Resources](https://github.com/cyruslayo/castalia)
- Explore related notebooks in the evals/ directory

---

## 🧭 Navigation

⬅️ **Previous:** [13 Tool Use And Task Success Eval](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/evals/13_tool_use_and_task_success_eval.ipynb) | ➡️ **Next:** [15 Multi Agent System Eval](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/evals/15_multi_agent_system_eval.ipynb)